In [ ]:

{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Análisis Exploratorio y Cálculo de KPIs sobre la Deserción en Educación Superior en Colombia\n",
    "\n",
    "**Proyecto ETL - Educación Superior Colombiana**  \n",
    "**Responsable:** Integrante 4 (Análisis, Validación y KPIs)  \n",
    "**Fuente:** Capa Gold (`data/gold/gold_final.csv`)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Configuración visual\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.figsize\"] = (12, 6)\n",
    "plt.rcParams[\"font.size\"] = 10\n",
    "pd.set_option(\"display.max_columns\", 50)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Carga del DataFrame Gold y verificación inicial"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ruta_gold = os.path.join(\"..\", \"data\", \"gold\", \"gold_final.csv\")\n",
    "\n",
    "if not os.path.exists(ruta_gold):\n",
    "    ruta_gold = os.path.join(\"data\", \"gold\", \"gold_final.csv\")\n",
    "\n",
    "if not os.path.exists(ruta_gold):\n",
    "    raise FileNotFoundError(\"No se encontró data/gold/gold_final.csv. Verifique la ruta o ejecute primero el pipeline ETL.\")\n",
    "\n",
    "df_gold = pd.read_csv(ruta_gold)\n",
    "\n",
    "print(f\"Filas cargadas: {df_gold.shape[0]:,}\")\n",
    "print(f\"Columnas cargadas: {df_gold.shape[1]}\")\n",
    "display(df_gold.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Tipos de datos y estructura"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_gold.info()\n",
    "\n",
    "print(\"\\nColumnas disponibles:\")\n",
    "print(df_gold.columns.tolist())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Validación de nulos, duplicados y claves"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=== VALORES NULOS POR COLUMNA ===\")\n",
    "print(df_gold.isnull().sum())\n",
    "\n",
    "print(\"\\n=== DUPLICADOS EXACTOS ===\")\n",
    "print(f\"Filas duplicadas: {df_gold.duplicated().sum()}\")\n",
    "\n",
    "print(\"\\n=== VALIDACIÓN DE CLAVES ===\")\n",
    "clave = [\"departamento_clave\", \"anio\"]\n",
    "\n",
    "if all(col in df_gold.columns for col in clave):\n",
    "    print(f\"Claves departamento-año duplicadas: {df_gold.duplicated(subset=clave).sum()}\")\n",
    "    print(f\"Departamentos sin identificar: {df_gold['departamento_clave'].isna().sum()}\")\n",
    "    print(f\"Años sin identificar: {df_gold['anio'].isna().sum()}\")\n",
    "else:\n",
    "    raise ValueError(\"La capa Gold no contiene las columnas esperadas para identificar departamento y año.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Estadísticas descriptivas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "display(df_gold.describe(include=\"all\").T)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Preparación de las tasas e indicadores clave (KPIs)\n",
    "\n",
    "La capa Gold contiene una columna de tasa para cada nivel de formación. Para facilitar la comparación, se transforma temporalmente la información a formato largo. Los valores faltantes se conservan y no se reemplazan por cero."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Columnas de tasas disponibles en Gold\n",
    "columnas_tasa = [\n",
    "    \"tasa_desercion_universitario\",\n",
    "    \"tasa_desercion_tyt\",\n",
    "    \"tasa_desercion_tecnico_profesional\",\n",
    "    \"tasa_desercion_tecnologico\"\n",
    "]\n",
    "\n",
    "columnas_tasa = [col for col in columnas_tasa if col in df_gold.columns]\n",
    "\n",
    "if not columnas_tasa:\n",
    "    raise ValueError(\"No se encontraron las columnas de tasas de deserción esperadas en Gold.\")\n",
    "\n",
    "id_vars = [col for col in [\"departamento_clave\", \"anio\", \"coincidio_matricula\"] if col in df_gold.columns]\n",
    "\n",
    "tasas_largas = df_gold.melt(\n",
    "    id_vars=id_vars,\n",
    "    value_vars=columnas_tasa,\n",
    "    var_name=\"nivel_formacion\",\n",
    "    value_name=\"tasa_desercion\"\n",
    ")\n",
    "\n",
    "nombres_niveles = {\n",
    "    \"tasa_desercion_universitario\": \"Universitario\",\n",
    "    \"tasa_desercion_tyt\": \"TyT\",\n",
    "    \"tasa_desercion_tecnico_profesional\": \"Técnico Profesional\",\n",
    "    \"tasa_desercion_tecnologico\": \"Tecnológico\"\n",
    "}\n",
    "\n",
    "tasas_largas[\"nivel_formacion\"] = tasas_largas[\"nivel_formacion\"].map(nombres_niveles)\n",
    "tasas_largas[\"tasa_desercion\"] = pd.to_numeric(tasas_largas[\"tasa_desercion\"], errors=\"coerce\")\n",
    "\n",
    "# Se excluyen únicamente los valores faltantes del cálculo, no de la fuente original.\n",
    "tasas_validas = tasas_largas.dropna(subset=[\"tasa_desercion\"]).copy()\n",
    "\n",
    "# Validación del rango de las tasas\n",
    "fuera_rango = ~tasas_validas[\"tasa_desercion\"].between(0, 1)\n",
    "if fuera_rango.any():\n",
    "    raise ValueError(\"Se encontraron tasas de deserción fuera del rango de 0 a 1.\")\n",
    "\n",
    "print(\"Columnas de tasas utilizadas:\")\n",
    "print(columnas_tasa)\n",
    "print(f\"Registros válidos para análisis: {len(tasas_validas):,}\")\n",
    "print(f\"Valores de tasa faltantes: {tasas_largas['tasa_desercion'].isna().sum():,}\")\n",
    "\n",
    "display(tasas_validas.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# KPI 1: Promedio simple de las tasas disponibles\n",
    "tasa_promedio = tasas_validas[\"tasa_desercion\"].mean()\n",
    "\n",
    "# KPI 2: Promedio simple por nivel de formación\n",
    "tasa_por_nivel = (\n",
    "    tasas_validas.groupby(\"nivel_formacion\")[\"tasa_desercion\"]\n",
    "    .agg([\"mean\", \"min\", \"max\", \"count\"])\n",
    "    .reset_index()\n",
    ")\n",
    "\n",
    "tasa_por_nivel[\"promedio_porcentaje\"] = tasa_por_nivel[\"mean\"] * 100\n",
    "tasa_por_nivel[\"minimo_porcentaje\"] = tasa_por_nivel[\"min\"] * 100\n",
    "tasa_por_nivel[\"maximo_porcentaje\"] = tasa_por_nivel[\"max\"] * 100\n",
    "\n",
    "print(f\"PROMEDIO SIMPLE DE TASAS DISPONIBLES: {tasa_promedio * 100:.2f}%\")\n",
    "print(\"\\nPROMEDIO SIMPLE POR NIVEL DE FORMACIÓN:\")\n",
    "display(tasa_por_nivel[[\"nivel_formacion\", \"promedio_porcentaje\", \"minimo_porcentaje\", \"maximo_porcentaje\", \"count\"]])\n",
    "\n",
    "print(\"\\nNota: estos promedios no están ponderados por cantidad de estudiantes y no representan necesariamente una tasa nacional oficial.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# KPI 3: Cobertura de coincidencia con la fuente de matrícula\n",
    "if \"coincidio_matricula\" in df_gold.columns:\n",
    "    cobertura_matricula = df_gold[\"coincidio_matricula\"].astype(\"boolean\").mean() * 100\n",
    "    print(f\"COBERTURA DE MATRÍCULA INTEGRADA EN GOLD: {cobertura_matricula:.2f}%\")\n",
    "else:\n",
    "    print(\"La columna coincidio_matricula no está disponible en Gold.\")\n",
    "\n",
    "# Periodo disponible\n",
    "print(f\"\\nPeriodo de años disponible: {int(df_gold['anio'].min())} - {int(df_gold['anio'].max())}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Visualizaciones estratégicas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gráfica 1: Evolución temporal de las tasas por nivel de formación\n",
    "evolucion = (\n",
    "    tasas_validas.groupby([\"anio\", \"nivel_formacion\"], as_index=False)[\"tasa_desercion\"]\n",
    "    .mean()\n",
    ")\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "sns.lineplot(\n",
    "    data=evolucion,\n",
    "    x=\"anio\",\n",
    "    y=\"tasa_desercion\",\n",
    "    hue=\"nivel_formacion\",\n",
    "    marker=\"o\"\n",
    ")\n",
    "plt.title(\"Evolución de la tasa promedio de deserción por nivel de formación\")\n",
    "plt.xlabel(\"Año\")\n",
    "plt.ylabel(\"Tasa de deserción\")\n",
    "plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f\"{x:.0%}\"))\n",
    "plt.legend(title=\"Nivel de formación\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gráfica 2: Diez departamentos con mayor promedio simple de tasas disponibles\n",
    "top_deptos = (\n",
    "    tasas_validas.groupby(\"departamento_clave\")[\"tasa_desercion\"]\n",
    "    .mean()\n",
    "    .nlargest(10)\n",
    "    .reset_index()\n",
    ")\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "sns.barplot(data=top_deptos, x=\"tasa_desercion\", y=\"departamento_clave\")\n",
    "plt.title(\"Departamentos con mayor promedio simple de tasas de deserción\")\n",
    "plt.xlabel(\"Promedio simple de la tasa\")\n",
    "plt.ylabel(\"Departamento\")\n",
    "plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f\"{x:.0%}\"))\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gráfica 3: Distribución de tasas por nivel de formación\n",
    "plt.figure(figsize=(11, 6))\n",
    "sns.boxplot(data=tasas_validas, x=\"nivel_formacion\", y=\"tasa_desercion\")\n",
    "plt.title(\"Distribución de las tasas de deserción por nivel de formación\")\n",
    "plt.xlabel(\"Nivel de formación\")\n",
    "plt.ylabel(\"Tasa de deserción\")\n",
    "plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f\"{x:.0%}\"))\n",
    "plt.xticks(rotation=15)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Resultados Clave (KR) Evaluados\n",
    "\n",
    "- **KR1 (Comparación por nivel de formación):** La capa Gold permite comparar las tasas reportadas para los niveles Universitario, TyT, Técnico Profesional y Tecnológico. Las diferencias observadas son descriptivas y no demuestran que el nivel de formación sea una causa de la deserción.\n",
    "\n",
    "- **KR2 (Diferencias territoriales):** Es posible identificar departamentos con promedios más altos o más bajos dentro del periodo y de los niveles que cuentan con datos. Las posiciones pueden variar según el año y el nivel analizado.\n",
    "\n",
    "- **KR3 (Tendencias principales):** La evolución anual permite observar cambios en las tasas reportadas durante los años disponibles. No se debe afirmar una reducción continua sin revisar la serie completa.\n",
    "\n",
    "- **KR4 (Integración de fuentes):** La capa Gold conserva la información de deserción de SPADIES y agrega la información de matrícula cuando existe coincidencia por departamento y año.\n",
    "\n",
    "- **KR5 (Alcance analítico):** La capa permite análisis descriptivos por departamento, año y nivel formativo. No permite conclusiones específicas por institución, programa o estudiante porque la fuente SPADIES utilizada no tiene ese nivel de detalle."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Conclusiones y Limitaciones Conocidas\n",
    "\n",
    "1. **Análisis descriptivo:** La capa Gold permite describir y comparar las tasas de deserción reportadas por departamento, año y nivel de formación.\n",
    "\n",
    "2. **Interpretación de resultados:** Las diferencias entre niveles o departamentos representan asociaciones descriptivas. Con estos datos no es posible establecer relaciones causales ni atribuir la deserción a una característica particular.\n",
    "\n",
    "3. **Granularidad de la fuente:** La información de deserción de SPADIES se encuentra agregada por departamento, nivel de formación y año. Por esta razón, el análisis no permite asignar tasas a instituciones o programas individuales.\n",
    "\n",
    "4. **Integración de matrícula:** La integración se realiza a nivel departamento-año. La coincidencia entre fuentes no implica que ambas midan exactamente la misma población ni que los registros de matrícula representen estudiantes únicos.\n",
    "\n",
    "5. **Cobertura temporal:** La API de matrícula disponible cubre el periodo 2015-2021, mientras que SPADIES contiene tasas entre 2010 y 2024. Por ello, no todos los registros de deserción tienen una coincidencia de matrícula.\n",
    "\n",
    "6. **Cálculo de promedios:** Los promedios presentados son simples y no están ponderados por la cantidad de estudiantes. Por tanto, no deben interpretarse como tasas nacionales oficiales.\n",
    "\n",
    "7. **Valores faltantes:** Los valores faltantes se conservan como datos no disponibles y no se interpretan como tasas iguales a cero."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
